<a href="https://colab.research.google.com/github/PavithraSivakumar-12/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PavithraSivakumar-12/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("✅ Connected to Hugging Face dataset")

✅ Connected to Hugging Face dataset


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
## Method Choice and Why

I selected a Decision Tree Classifier for this task.

The goal of this lane is to identify pages with high impressions but relatively low clicks so they can be reviewed. A Decision Tree can learn simple decision rules from search metrics and is easy to interpret. It also provides a fair comparison against the Week 4 baseline rule while remaining simple enough to explain.

The model uses observed search performance features and is intended for decision support rather than proving causation.

In [1]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

print(model)

DecisionTreeClassifier(max_depth=5, random_state=42)


## 2. Split design

## Split Design

An 80/20 train-test split is used for this model.

The training data is used to fit the Decision Tree model, while the test data is used only for evaluation. The same split is used when comparing the model against the Week 4 baseline so the comparison is fair.

This split supports an honest evaluation on unseen data and is intended for decision-support analysis.

In [4]:
from sklearn.model_selection import train_test_split

rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE gsc_data_available IS TRUE
""").df()

# Create the same target used in the baseline
df["target"] = (
    (df["gsc_impressions"] > 1000) &
    (df["gsc_clicks"] < 10)
).astype(int)

X = df[["gsc_impressions", "gsc_clicks", "gsc_sum_position"]]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training rows: 23176040
Testing rows : 5794011


## 3. Train + compare vs my baseline

## Train + Compare vs My Baseline

A Decision Tree classifier is trained using the selected search performance features.

The model is evaluated on the same train-test split used throughout this notebook. The baseline predicts the majority class, while the Decision Tree learns patterns from the data.

The comparison uses accuracy on the same test set.

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Baseline model
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

# Decision Tree model
model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

baseline_acc = accuracy_score(y_test, baseline_pred)
model_acc = accuracy_score(y_test, model_pred)

results = pd.DataFrame({
    "Model": ["Baseline", "Decision Tree"],
    "Accuracy": [baseline_acc, model_acc]
})

print(results)

           Model  Accuracy
0       Baseline  0.994685
1  Decision Tree  1.000000


## 4. Errors and interpretation

## Errors and Interpretation

The Decision Tree achieved higher accuracy than the baseline on the same test split.

The confusion matrix shows that all observations in the test set were classified correctly. This perfect result is likely because the target label was created directly from the same search metrics (impressions and clicks) that were used as model features.

Therefore, this notebook demonstrates how the model learns the defined rule rather than proving that it will generalize to unseen real-world data.

The model should be interpreted as decision-support and not as evidence of causal relationships. No future-window information or product-generated labels were used.

In [7]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, model_pred)

print("Confusion Matrix")
print(cm)

print("\nModel Accuracy:", round(model_acc, 4))
print("Baseline Accuracy:", round(baseline_acc, 4))

Confusion Matrix
[[5763217       0]
 [      0   30794]]

Model Accuracy: 1.0
Baseline Accuracy: 0.9947


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.